# ANÁLISIS EXPLORATORIO DE DATOS (EDA)

En esta sección ejecutaremos el análisis estadístico descriptivo e inferencial sobre el dataset final procesado. Utilizaremos exclusivamente **Pandas** y **NumPy** para extraer los estadísticos clave y evaluar las correlaciones entre el rendimiento en pista, la estrategia y el resultado final de carrera.


## 1. Carga del Dataset Procesado y Cálculo de Estadísticos Descriptivos

Analizamos las medidas de tendencia central (media y mediana) y dispersión (desviación estándar, mínimos y máximos) para las variables cuantitativas principales.

In [1]:
import pandas as pd
import numpy as np

# Cargar el conjunto de datos
df = pd.read_csv('f1_data.csv')
df.head()

/var/folders/9m/tm3_26dd11sfl1btdhw5wkwh0000gn/T/ipykernel_54680/3573758656.py:5: DtypeWarning: Columns (11,15,19,28,44) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('f1_data.csv')


,raceId,driverId,lap,position_lap,time_lap,milliseconds_lap,resultId,constructorId,number,grid,...,alt,stop,pit_duration,pit_milliseconds,driver_full_name,driver_age,lap_seconds,positions_gained,is_pit_stop,pit_seconds
0,479,137,1,1,1:42.085,102085,11437,34,1.0,1.0,...,678,NaN,NaN,NaN,Nelson Piquet,30,102.085,-10.0,False,NaN
1,479,137,2,2,1:36.287,96287,11437,34,1.0,1.0,...,678,NaN,NaN,NaN,Nelson Piquet,30,96.287,-10.0,False,NaN
2,479,137,3,2,1:34.627,94627,11437,34,1.0,1.0,...,678,NaN,NaN,NaN,Nelson Piquet,30,94.627,-10.0,False,NaN
3,479,137,4,2,1:34.041,94041,11437,34,1.0,1.0,...,678,NaN,NaN,NaN,Nelson Piquet,30,94.041,-10.0,False,NaN
4,479,137,5,2,1:33.699,93699,11437,34,1.0,1.0,...,678,NaN,NaN,NaN,Nelson Piquet,30,93.699,-10.0,False,NaN


In [2]:
# 1. Selección de columnas cuantitativas principales 
cols_eda = ['lap_seconds', 'driver_age', 'grid', 'positionOrder', 'positions_gained', 'pit_seconds']

# 2. Cáaculo de estadísticos descriptivos
estadisticos = df[cols_eda].describe().T[['count', 'mean', 'std', 'min', '50%', 'max']].round(2)
estadisticos.columns = ['Recuento', 'Media', 'Desv. Est.', 'Mínimo', 'Mediana (Q2)', 'Máximo']

print(estadisticos)

                  Recuento  Media  Desv. Est.  Mínimo  Mediana (Q2)   Máximo
lap_seconds       278678.0  95.64       78.12   65.26         90.71  7507.55
driver_age        278678.0  29.89        4.44   20.00         30.00    43.00
grid              278678.0  12.63        7.42    1.00         12.00    29.00
positionOrder     278678.0   9.99        6.04    1.00          9.00    27.00
positions_gained  278678.0   2.64        7.10  -25.00          3.00    22.00
pit_seconds         2424.0  30.14       30.42   12.90         25.50  1004.72


In [3]:
# Análisis del factor "Pole Position a Victoria" 
races_unicas = df[['raceId', 'driverId', 'grid', 'positionOrder']].drop_duplicates()

total_poles = np.sum(races_unicas['grid'] == 1)
total_pole_wins = np.sum((races_unicas['grid'] == 1) & (races_unicas['positionOrder'] == 1))
pct_conversion = (total_pole_wins / total_poles) * 100

print(f"Carreras con Pole analizadas: {total_poles}")
print(f"Victorias desde la Pole Position: {total_pole_wins}")
print(f"Porcentaje de conversión a victoria: {pct_conversion}%")

Carreras con Pole analizadas: 244
Victorias desde la Pole Position: 97
Porcentaje de conversión a victoria: 39.75409836065574%


## 2.Análisis de interés

A continuación dmaos un paso más dentro del estudio del conjunto de datos que disponemos. Ampliamos el análisis exploratorio ejecutando cuatro estudios estadísticos independientes sobre el dataset unificado (`f1_data.csv`), los cuales responden a preguntas muy interesantes que pueden describir y relacionar el rendimiento con los diferentes factores, no solo operacionales, que componen el dataset.

1. **Estudio de Remontadas:** Agregación de posiciones ganadas netas por piloto.
2. **Análisis de Fiabilidad (DNFs):** Tasa de abandonos por escudería usando filtros.
3. **Percentiles Operativos de Boxes:** Cálculo de $P_{25}$, $P_{50}$ y $P_{75}$ en paradas en boxes.

### ESTUDIO 1: ANÁLISIS DE REMONTADAS Y RECUPERACIÓN DE POSICIONES 

In [4]:
remontadas = df[['raceId', 'driver_full_name', 'positions_gained', 'grid']].drop_duplicates()
analisis_remontadas = remontadas.groupby('driver_full_name').agg(
    carreras=('positions_gained', 'count'),
    media_pos_ganadas=('positions_gained', 'mean'),
    max_remontada=('positions_gained', 'max'),
    std_remontada=('positions_gained', 'std')
).query('carreras >= 30').sort_values('media_pos_ganadas', ascending=False).head(5).round(2)

print(analisis_remontadas)


                      carreras  media_pos_ganadas  max_remontada  \
driver_full_name                                                   
Jonathan Palmer             81               6.59           19.0   
Christian Fittipaldi        40               6.35           16.0   
Christian Danner            34               6.21           22.0   
Marc Surer                  55               5.22           20.0   
Gabriele Tarquini           35               5.17           18.0   

                      std_remontada  
driver_full_name                     
Jonathan Palmer                6.68  
Christian Fittipaldi           6.32  
Christian Danner               6.51  
Marc Surer                     7.66  
Gabriele Tarquini              6.77  


 #### Interpretación y Conclusión  — Estudio 1 (Remontadas)
 
 * **Efecto de la Parrilla Media/Baja:** Se observa que los pilotos que acumulan un mayor promedio de posiciones ganadas (`positions_gained`) parten habitualmente de posiciones intermedias o traseras de la parrilla (`grid > 10`). Por la experiencia de ver diferentes carreras a lo largo de los año, y sin ser una experta, esto puede deberse a que la zona media de la parrilla presenta una mayores cambios debido a incidentes en la primera vuelta y dispersión de estrategias de neumáticos.
 * **Rango de Varianza ($\sigma$):** La elevada desviación estándar ($\sigma \approx 7.42$) evidencia que la capacidad de remontada no es constante, sino altamente dependiente de factores externos como banderas rojas, coches de seguridad (*Safety Car*) y condiciones meteorológicas cambiantes.
* **Picos Históricos:** El registro de un valor máximo de **+22 posiciones ganadas** demuestra que salir en los últimos puestos no elimina la probabilidad de puntuar cuando se ejecuta una estrategia limpia. 

### ESTUDIO 2: FIABILIDAD Y TASA DE ABANDONOS (DNFs) POR ESCUDERÍA 
Nota para este eestudio, hubo un problema interesante, en un principio se pensaba que  statusId == 1 representa coche clasificado/finalizado, sin embargo tras hacer el estudio teniendo esto en cuenta los resultados y porcentajes resultantes no tenían nada de sentido. Después de una investigación del conjunto de datops y ver la fuente oficial se entendió que el filtro statusId != 1 no equivale a "Abandono", si no que: 

* statusId == 1 significa estrictamente "Finished" (cruzar la meta en la misma vuelta que el ganador).

* statusId == 11, 12, 13, etc. significan "+1 Lap", "+2 Laps", "+3 Laps" (pilotos que SÍ terminaron la carrera, pero doblados por el líder).

Lo que estaba hacuendo al inicio, estaba contando a todos los pilotos clasificados que quedaron doblados como si se hubiesen estrellado o roto el motor. Por los porcentajes eran desorbitados de abandonos, por ehemplo los de McLaren/Williams con un 46%-55%.


Para medir la fiabilidad real de las escuderías, debemos utilizar la tabla de resultados completa (results.csv) e identificar como "Abandono/DNF" solo los estados que corresponden a accidentes, averías o descalificaciones

*Consideraciones*
1. **Desduplicación a Nivel de Carrera:** Dado que `f1_data` contiene una fila por cada vuelta disputada, utilizamos `.drop_duplicates(subset=['raceId', 'driverId'])` para reducir el conjunto a una sola fila por participación de piloto en cada Gran Premio.
2. **Finalización bien interpretada:** Definimos la lista `[1, 11, 12, 13, 14, 15, 16, 17, 18, 19]` para marcar como `True` a todos los vehículos que completaron la carrera (tanto líderes como doblados) y contabilizar como abandonos únicamente las interrupciones reales por avería o accidente.


In [23]:
# 1. Cargar datos en bruto con nulos estandarizados para facilitar el estudio 
results = pd.read_csv('rawdata/results.csv', na_values='\\N')
constructors = pd.read_csv('rawdata/constructors.csv', na_values='\\N')
status = pd.read_csv('rawdata/status.csv', na_values='\\N')

# 2. Unir resultados con la dimensión de escuderías
df_fiabilidad = results.merge(constructors, on='constructorId')

# 3. Identificar estados de FINALIZACIÓN REAL 
# statusId en [1, 11, 12, 13, 14, 15, 16, 17, 18, 19] corresponden a coches que TERMINARON la carrera
estados_finalizados = [1, 11, 12, 13, 14, 15, 16, 17, 18, 19]

# 4. Calcular métricas por escudería
resumen_fiabilidad = df_fiabilidad.groupby('name').agg(total_participaciones=('resultId', 'count'),
    abandonos=('statusId', lambda x: np.sum(~x.isin(estados_finalizados))))

resumen_fiabilidad['porcentaje_abandonos'] = (resumen_fiabilidad['abandonos'] / resumen_fiabilidad['total_participaciones'] * 100).round(2)

# Filtrar escuderías con volumen mínimo histórico (>= 100 participaciones de coches)
top_fiabilidad = resumen_fiabilidad.query('total_participaciones >= 100').sort_values('porcentaje_abandonos')

print(top_fiabilidad.head(20))

                total_participaciones  abandonos  porcentaje_abandonos
name                                                                  
Mercedes                          724         86                 11.88
RB F1 Team                        120         16                 13.33
Alpine F1 Team                    252         36                 14.29
Marussia                          109         17                 15.60
AlphaTauri                        166         27                 16.27
BMW Sauber                        140         23                 16.43
Red Bull                          860        149                 17.33
Haas F1 Team                      452         85                 18.81
Aston Martin                      262         51                 19.47
Force India                       424         88                 20.75
Caterham                          112         25                 22.32
Lotus F1                          154         42                 27.27
Toro R

#### Interpretación y Conclusión — Estudio 2 (Fiabilidad Mecánica)
 
 * **Evolución Tecnológica e Ingeniería:** Se aprecia una brecha estructural de fiabilidad entre las distintas eras de la Fórmula 1. Escuderías modernas como **Mercedes (11.88% de abandonos)** y **Red Bull (17.33%)** presentan tasas de fallo significativamente inferiores a las históricas.
 * **Sesgo del Registro Histórico:** Escuderías legendarias como **Ferrari (30.11 de DNFs)** y **McLaren (30.33%)** muestran una tasa de abandonos cercana al 35%. Esto no implica una deficiente ingeniería moderna, sino el peso del histórico acumulado desde 1950, época en la que la tasa de fallos mecánicos era superior.
 * **Impacto en Negocio/Estrategia:** La fiabilidad actual supera el 80% en la parrilla, lo que exige que las victorias se decidan por milésimas en ritmo de carrera y precisión en boxes, a diferencia de décadas pasadas.

### ESTUDIO 3: PERCENTILES Y DISTRIBUCIÓN EN PIT STOPS CON NUMPY 

In [36]:
# Cálculo de Estadísticos (Media + Percentiles)
media = pit_vueltas.mean().round(2)
p25 = np.percentile(pit_vueltas, 25).round(2)
p50 = np.percentile(pit_vueltas, 50).round(2)  # Mediana
p75 = np.percentile(pit_vueltas, 75).round(2)

print(f"Media (promedio inflado por outliers): {media} s")
print(f"Percentil 25 (Parada rápida): {p25} s")
print(f"Percentil 50 - Mediana (Parada típica): {p50} s")
print(f"Percentil 75 (Parada lenta): {p75} s")

Media (promedio inflado por outliers): 30.14 s
Percentil 25 (Parada rápida): 22.23 s
Percentil 50 - Mediana (Parada típica): 25.5 s
Percentil 75 (Parada lenta): 31.43 s


#### Interpretación y Conclusión — Estudio 4 (Eficiencia en Boxes)
 
 * **Presencia de Sesgo por Outliers:** La media matemática de **30.14 s** frente a la mediana de **25.50 s** evidencia una distribución asimétrica positiva (sesgada a la derecha). Esto confirma que para modelar el tiempo típico de una parada en boxes en Power BI o Python debemos emplear la mediana ($P_{50}$) como indicador de tendencia central, ya que la media está  inflada por paradas con incidencias o reparaciones.

 * **Rango Intercuartílico ($P_{75} - P_{25} = 9.20\text{ s}$):** La ventana operativa estándar entre una parada rápida ($22.23\text{ s}$) y una parada lenta con tráfico ($31.43\text{ s}$) es de 9.20 segundos. En el contexto estratégico de la F1, esta ventana determina el éxito de un *undercut* (adelantamiento en boxes) o la pérdida neta de posiciones en pista.